In [25]:
import os
import glob
import cv2
import numpy as np
from collections import defaultdict
from tqdm import tqdm
import torch
import numpy as np

def imread_korean(path, flags=cv2.IMREAD_COLOR):
    try:
        img_array = np.fromfile(path, np.uint8)
        img = cv2.imdecode(img_array, flags)
        return img
    except Exception as e:
        print(f"이미지 읽기 오류: {e}")
        return None

def imwrite_korean(path, img, params=None):
    try:
        ext = os.path.splitext(path)[1]
        result, encoded_img = cv2.imencode(ext, img, params)
        
        if result:
            with open(path, mode='w+b') as f:
                encoded_img.tofile(f)
            return True
        else:
            return False
    except Exception as e:
        print(f"이미지 저장 오류: {e}")
        return False

In [23]:
def scan_mask_folder(folder_path):
    
    mask_files = glob.glob(os.path.join(folder_path, '*.png'))


    print(f"{len(mask_files)} files \n")
    
    total_pixel_counts = defaultdict(int)
    
    for mask_path in tqdm(mask_files):
        mask = imread_korean(mask_path, cv2.IMREAD_GRAYSCALE)
        
        if mask is None:
            continue
            
        unique_vals, counts = np.unique(mask, return_counts=True)
        
        for val, count in zip(unique_vals, counts):
            total_pixel_counts[val] += count
    
    for val in sorted(total_pixel_counts.keys()):
        class_name = f"Class {val}" if val != 255 else "Class 255 (Ignore)"
        print(f" {class_name:18}: {total_pixel_counts[val]:>15,} pixel")

FOLDER_PATH = r'D:\Study\HumanStudy\Dataset\Training_Masks'

scan_mask_folder(FOLDER_PATH)

98398 files 



100%|████████████████████████████████████████████████████████████████████████████| 98398/98398 [41:19<00:00, 39.68it/s]

 Class 0           : 189,813,019,719 pixel
 Class 1           :     290,496,508 pixel
 Class 2           :   8,195,387,871 pixel
 Class 3           :   3,147,118,299 pixel
 Class 4           :     496,204,801 pixel
 Class 5           :     285,922,407 pixel
 Class 6           :     262,533,159 pixel
 Class 7           :   1,392,698,324 pixel
 Class 8           :       8,313,224 pixel
 Class 9           :     116,371,158 pixel
 Class 10          :      30,027,330 pixel


In [39]:
pixel_counts = np.array([
    189813019719, # 0: Background
    290496508,    # 1: Crack
    8195387871,   # 2: Leak
    3147118299,   # 3: Efflorescence
    496204801,    # 4: Detachment
    285922407,    # 5: Reticular crack
    262533159,    # 6: Spalling
    1392698324,   # 7: Material separation
    8313224,      # 8: Rebar
    116371158,    # 9: Damage
    30027330      # 10: Exhilaration
], dtype=np.float64)

total_pixels = np.sum(pixel_counts)
probabilities = pixel_counts / total_pixels
c = 1.04 
weights = 1.0 / np.log(c + probabilities)

print("weights = torch.tensor([")
for i, w in enumerate(weights):
    print(f"    {w:.4f},  # Class {i}")
print("], dtype=torch.float32).to(device)")

weights = torch.tensor([
    1.4745,  # Class 0
    24.6374,  # Class 1
    12.9677,  # Class 2
    18.5382,  # Class 3
    24.0637,  # Class 4
    24.6504,  # Class 5
    24.7175,  # Class 6
    21.8520,  # Class 7
    25.4713,  # Class 8
    25.1452,  # Class 9
    25.4051,  # Class 10
], dtype=torch.float32).to(device)
